[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hany73-max/PDE-Rover-Engine/blob/main/labs_ar/Lab_1_Elimination_of_Constants_AR.ipynb) [![Launch JupyterLab](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/hany73-max/PDE-Rover-Engine/main?urlpath=lab/tree/labs_ar/Lab_1_Elimination_of_Constants_AR.ipynb)


In [ ]:
# 🚀 تهيئة مسار المشروع تلقائياً (يعمل على JupyterLab المحلي، وBinder، وGoogle Colab)
import os, sys
try:
    if 'google.colab' in str(get_ipython()):
        if not os.path.exists('PDE-Rover-Engine'):
            import subprocess
            subprocess.run(['git', 'clone', 'https://github.com/hany73-max/PDE-Rover-Engine.git'], check=True)
        else:
            import subprocess
            subprocess.run(['git', '-C', 'PDE-Rover-Engine', 'pull', '--ff-only'], check=False)
        os.chdir('PDE-Rover-Engine')
except NameError:
    pass

# التأكد من إضافة المجلد الرئيسي للمشروع (الذي يحتوي على labs/ و rover_path_finder/) إلى مسار بايثون
# سواء تم تشغيل النوت بوك من المجلد الرئيسي أو من داخل labs_ar/ أو solutions/labs_ar/
_cwd = os.path.abspath(os.getcwd())
for _candidate in [_cwd, os.path.abspath(os.path.join(_cwd, '..')), os.path.abspath(os.path.join(_cwd, '..', '..'))]:
    if os.path.isdir(os.path.join(_candidate, 'labs', 'tests')) and os.path.isdir(os.path.join(_candidate, 'rover_path_finder')):
        if _candidate not in sys.path:
            sys.path.insert(0, _candidate)
        break
print('[OK] تم تهيئة مسار المشروع بنجاح!')


<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; width:100%; max-width:100%; box-sizing:border-box; line-height:1.85; padding:6px 14px 16px 14px;">

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:12px 0 6px 0;">

# ‏📡 اللاب 1: تكوين المعادلات التفاضلية الجزئية بحذف الثوابت الاختيارية

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:6px 0; line-height:1.85;">

‏<b>المقرر:</b> المعادلات التفاضلية الجزئية وتطبيقاتها في ملاحة المركبات الفضائية

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:6px 0; line-height:1.85;">

‏<b>المرجع في المذكرة:</b> الباب الأول، القسم 1.1 و 1.2 (صفحات المذكرة 1–10 / صفحات ملف PDF: 2–11)

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; border-right:4px solid #f59e0b; background-color:rgba(245,158,11,0.08); padding:10px 14px; margin:12px 0; border-radius:4px; line-height:1.85;">

‏🗺️ <b>أين يقع هذا اللاب في خط عمل المشروع؟</b><br>
‏‏<code dir="rtl" style="direction:rtl; display:inline-block; unicode-bidi:isolate; padding:1px 5px; border-radius:3px;">خريطة الارتفاعات h(x,y)</code>‏ ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${\longrightarrow}$</span>‏ <b>‏<code dir="rtl" style="direction:rtl; display:inline-block; unicode-bidi:isolate; padding:1px 5px; border-radius:3px;">[اللاب 1: حساب الميول p, q وكشف العوائق slope&#95;map]</code>‏</b> ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${\longrightarrow}$</span>‏ ‏<code dir="rtl" style="direction:rtl; display:inline-block; unicode-bidi:isolate; padding:1px 5px; border-radius:3px;">نطاق الأمان</code>‏ ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${\longrightarrow}$</span>‏ ‏<code dir="rtl" style="direction:rtl; display:inline-block; unicode-bidi:isolate; padding:1px 5px; border-radius:3px;">معادلة لابلاس</code>‏ ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${\longrightarrow}$</span>‏ ‏<code dir="rtl" style="direction:rtl; display:inline-block; unicode-bidi:isolate; padding:1px 5px; border-radius:3px;">مسار المركبة</code>‏<br>
‏<b>الملف الحقيقي في المشروع:</b> <a href="../rover_path_finder/terrain.py">‏<code dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate; padding:1px 5px; border-radius:3px;">rover&#95;path&#95;finder/terrain.py</code>‏</a> — الدالة ‏<code dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate; padding:1px 5px; border-radius:3px;">slope&#95;map(h, cell)</code>‏

</div>

</div>


<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; width:100%; max-width:100%; box-sizing:border-box; line-height:1.85; padding:6px 14px 16px 14px;">

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:12px 0 6px 0;">

### ‏🧭 قاموس الربط: الرياضيات ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${\longleftrightarrow}$</span>‏ المعنى الفيزيائي ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${\longleftrightarrow}$</span>‏ كود المشروع

</div>

<div dir="rtl" style="direction:rtl; text-align:right; overflow-x:auto; width:100%; margin:14px 0;">
<table dir="rtl" style="display:table; width:100%; border-collapse:collapse; direction:rtl; text-align:right;">
<thead>
<tr>
<th dir="rtl" align="right" style="direction:rtl; text-align:right; padding:9px 12px; border:1px solid rgba(128,128,128,0.4); background-color:rgba(128,128,128,0.16); font-weight:bold; line-height:1.75; vertical-align:middle;">

‏1. الرمز في مذكرة المحاضرة

</th>
<th dir="rtl" align="right" style="direction:rtl; text-align:right; padding:9px 12px; border:1px solid rgba(128,128,128,0.4); background-color:rgba(128,128,128,0.16); font-weight:bold; line-height:1.75; vertical-align:middle;">

‏2. المعنى الفيزيائي على أرض المريخ

</th>
<th dir="rtl" align="right" style="direction:rtl; text-align:right; padding:9px 12px; border:1px solid rgba(128,128,128,0.4); background-color:rgba(128,128,128,0.16); font-weight:bold; line-height:1.75; vertical-align:middle;">

‏3. الكود الفعلي في ‏<code dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate; padding:1px 5px; border-radius:3px;">terrain.py</code>‏

</th>
</tr>
</thead>
<tbody>
<tr>
<td dir="rtl" align="right" style="direction:rtl; text-align:right; padding:8px 12px; border:1px solid rgba(128,128,128,0.35); vertical-align:middle; line-height:1.8;">

‏‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate; white-space:nowrap;">${z = h(x, y)}$</span>‏

</td>
<td dir="rtl" align="right" style="direction:rtl; text-align:right; padding:8px 12px; border:1px solid rgba(128,128,128,0.35); vertical-align:middle; line-height:1.8;">

‏ارتفاع سطح الأرض (بالمتر) عند الإحداثيات ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate; white-space:nowrap;">${(x, y)}$</span>‏

</td>
<td dir="rtl" align="right" style="direction:rtl; text-align:right; padding:8px 12px; border:1px solid rgba(128,128,128,0.35); vertical-align:middle; line-height:1.8;">

‏مصفوفة الارتفاعات ‏<code dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate; padding:1px 5px; border-radius:3px;">h</code>‏ ثنائية الأبعاد

</td>
</tr>
<tr>
<td dir="rtl" align="right" style="direction:rtl; text-align:right; padding:8px 12px; border:1px solid rgba(128,128,128,0.35); vertical-align:middle; line-height:1.8;">

‏‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate; white-space:nowrap;">${p = \frac{\partial z}{\partial x}, \quad q = \frac{\partial z}{\partial y}}$</span>‏

</td>
<td dir="rtl" align="right" style="direction:rtl; text-align:right; padding:8px 12px; border:1px solid rgba(128,128,128,0.35); vertical-align:middle; line-height:1.8;">

‏ميل السطح باتجاه الشرق (‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate; white-space:nowrap;">${x}$</span>‏) وباتجاه الشمال (‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate; white-space:nowrap;">${y}$</span>‏)

</td>
<td dir="rtl" align="right" style="direction:rtl; text-align:right; padding:8px 12px; border:1px solid rgba(128,128,128,0.35); vertical-align:middle; line-height:1.8;">

‏‏<code dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate; padding:1px 5px; border-radius:3px;">gy, gx = np.gradient(h, cell)</code>‏

</td>
</tr>
<tr>
<td dir="rtl" align="right" style="direction:rtl; text-align:right; padding:8px 12px; border:1px solid rgba(128,128,128,0.35); vertical-align:middle; line-height:1.8;">

‏‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate; white-space:nowrap;">${\sqrt{p^2 + q^2}}$</span>‏

</td>
<td dir="rtl" align="right" style="direction:rtl; text-align:right; padding:8px 12px; border:1px solid rgba(128,128,128,0.35); vertical-align:middle; line-height:1.8;">

‏مقدار الانحدار الكلي للسطح (ظل زاوية الميل ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate; white-space:nowrap;">${\tan\theta}$</span>‏)

</td>
<td dir="rtl" align="right" style="direction:rtl; text-align:right; padding:8px 12px; border:1px solid rgba(128,128,128,0.35); vertical-align:middle; line-height:1.8;">

‏‏<code dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate; padding:1px 5px; border-radius:3px;">np.hypot(gx, gy)</code>‏ داخل ‏<code dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate; padding:1px 5px; border-radius:3px;">slope&#95;map(h)</code>‏

</td>
</tr>
<tr>
<td dir="rtl" align="right" style="direction:rtl; text-align:right; padding:8px 12px; border:1px solid rgba(128,128,128,0.35); vertical-align:middle; line-height:1.8;">

‏‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate; white-space:nowrap;">${F(x, y, z, a, b) = 0}$</span>‏

</td>
<td dir="rtl" align="right" style="direction:rtl; text-align:right; padding:8px 12px; border:1px solid rgba(128,128,128,0.35); vertical-align:middle; line-height:1.8;">

‏عائلة أسطح (مثلاً حفر بمراكز مختلفة ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate; white-space:nowrap;">${a, b}$</span>‏)

</td>
<td dir="rtl" align="right" style="direction:rtl; text-align:right; padding:8px 12px; border:1px solid rgba(128,128,128,0.35); vertical-align:middle; line-height:1.8;">

‏‏<code dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate; padding:1px 5px; border-radius:3px;">Z = (X - a)&#42;&#42;2 + (Y - b)&#42;&#42;2</code>‏

</td>
</tr>
<tr>
<td dir="rtl" align="right" style="direction:rtl; text-align:right; padding:8px 12px; border:1px solid rgba(128,128,128,0.35); vertical-align:middle; line-height:1.8;">

‏‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate; white-space:nowrap;">${f(x, y, z, p, q) = 0}$</span>‏

</td>
<td dir="rtl" align="right" style="direction:rtl; text-align:right; padding:8px 12px; border:1px solid rgba(128,128,128,0.35); vertical-align:middle; line-height:1.8;">

‏<b>المعادلة التفاضلية الجزئية (PDE):</b> قانون يربط الميل بالارتفاع بغض النظر عن قيمة ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate; white-space:nowrap;">${(a,b)}$</span>‏

</td>
<td dir="rtl" align="right" style="direction:rtl; text-align:right; padding:8px 12px; border:1px solid rgba(128,128,128,0.35); vertical-align:middle; line-height:1.8;">

‏‏<code dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate; padding:1px 5px; border-radius:3px;">residual = (P&#42;&#42;2 + Q&#42;&#42;2) - 4&#42;Z</code>‏

</td>
</tr>
</tbody>
</table>
</div>

</div>


<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; width:100%; max-width:100%; box-sizing:border-box; line-height:1.85; padding:6px 14px 16px 14px;">

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:12px 0 6px 0;">

## ‏1. الفكرة ببساطة: لماذا نحذف الثوابت الاختيارية؟

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:6px 0; line-height:1.85;">

‏تخيل أن على سطح المريخ مئات الحفر الدائرية. كل حفرة لها مركز مختلف ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${(a, b)}$</span>‏ على الخريطة:

</div>

<div dir="ltr" align="center" style="direction:ltr; text-align:center; margin:12px 0; overflow-x:auto;">

$$z = (x - a)^2 + (y - b)^2$$

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:6px 0; line-height:1.85;">

‏عندما يفحص حاسوب المركبة خريطة الارتفاعات ليقرر: <b>«هل هذه النقطة شديدة الانحدار (أكبر من ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${20^\circ}$</span>‏)؟»</b>، فإنه لا يعرف أين يقع مركز كل حفرة ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${(a, b)}$</span>‏ مسبقاً.

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:6px 0; line-height:1.85;">

‏<b>حذف الثوابت الاختيارية (Elimination of Arbitrary Constants)</b> يحوّل معادلة تعتمد على الموقع الخاص ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${(a, b)}$</span>‏ إلى <b>علاقة عامة تربط الميل ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${(p, q)}$</span>‏ بالعمق ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${z}$</span>‏ مباشرةً</b>:

</div>

<div dir="ltr" align="center" style="direction:ltr; text-align:center; margin:12px 0; overflow-x:auto;">

$$p^2 + q^2 = 4z$$

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:6px 0; line-height:1.85;">

‏هذه المعادلة تخبرنا بحقيقة هندسية رائعة: <b>مقدار ميل جدار الحفرة يعتمد فقط على عمق النقطة ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${z}$</span>‏، ولا يعتمد إطلاقاً على مكان الحفرة ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${(a, b)}$</span>‏!</b> ولهذا السبب تعمل دالة ‏<code dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate; padding:1px 5px; border-radius:3px;">slope&#95;map</code>‏ في مشروعنا بكفاءة على أي تضاريس دون معرفة معادلاتها أو مراكز الحفر فيها.

</div>

</div>


In [ ]:
import os, sys
for _p in ['.', '..', '../..']:
    _ap = os.path.abspath(_p)
    if os.path.isdir(os.path.join(_ap, 'labs', 'tests')) and _ap not in sys.path:
        sys.path.insert(0, _ap)
import numpy as np
import matplotlib.pyplot as plt
from labs.tests import test_lab1

%matplotlib inline
print("تم تهيئة بيئة اللاب 1 بنجاح!")


<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; width:100%; max-width:100%; box-sizing:border-box; line-height:1.85; padding:6px 14px 16px 14px;">

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:12px 0 6px 0;">

## ‏2. أساسيات القسم (1.1) وقاعدة عمل القسم (1.2) من المذكرة (PDF ص 2–11)

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:6px 0; line-height:1.85;">

‏قبل أن نحذف الثوابت، لنتقن المفاهيم الأساسية في <b>القسم (1.1) (صفحات المذكرة 1–5 / PDF ص 2–6)</b> لأنها تأتي دائماً في أسئلة الاختيارات والامتحانات:

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:12px 0 6px 0;">

### ‏أولاً: رموز مونج الخمسة (Monge's Notations — صفحة 3 بالمذكرة / PDF ص 4)

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:6px 0; line-height:1.85;">

‏في كل صفحات المذكرة، عندما تكون ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${z(x, y)}$</span>‏ دالة في متغيرين مستقلين ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${x}$</span>‏ و ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${y}$</span>‏، نستخدم الرموز القياسية التالية:

</div>

<div dir="ltr" align="center" style="direction:ltr; text-align:center; margin:12px 0; overflow-x:auto;">

$$p = \frac{\partial z}{\partial x}, \qquad q = \frac{\partial z}{\partial y}, \qquad r = \frac{\partial^2 z}{\partial x^2}, \qquad s = \frac{\partial^2 z}{\partial x \partial y}, \qquad t = \frac{\partial^2 z}{\partial y^2}$$

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:6px 0; line-height:1.85;">

‏<i>(ملاحظة جميلة للربط بالمشروع: في اللاب 4 واللاب 6 سنرى أن <b>معادلة لابلاس</b> للملاحة هي ببساطة ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${r + t = 0}$</span>‏!)</i>

</div>

</div>


<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; width:100%; max-width:100%; box-sizing:border-box; line-height:1.85; padding:6px 14px 16px 14px;">

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:12px 0 6px 0;">

### ‏ثانياً: الفرق بين الرتبة (Order) والدرجة (Degree) — والفخ الشهير! (ص 2 بالمذكرة / PDF ص 3)

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:7px 0; padding-right:8px; line-height:1.85;">

‏<b>1.</b> <b>الرتبة (Order):</b> هي رتبة <b>أعلى مشتقة جزئية</b> موجودة في المعادلة.

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:7px 0; padding-right:8px; line-height:1.85;">

‏<b>2.</b> <b>الدرجة (Degree):</b> هي <b>أُس (قوة) أعلى مشتقة جزئية</b>، ولكن <b>بشرط جوهري:</b> يجب أولاً التخلص من أي جذور أو أسس كسرية أو كسور تحتوي على المشتقات!

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:5px 0; padding-right:34px; line-height:1.85;">

‏▪ <b>🔍 مثال الفخ في المذكرة (مثال 5 ص 1–2 / PDF ص 2–3):</b>

</div>

<div dir="ltr" align="center" style="direction:ltr; text-align:center; margin:12px 0; overflow-x:auto;">

$$\frac{\partial^2 z}{\partial x^2} = \left(1 + \frac{\partial z}{\partial y}\right)^{\frac{1}{2}}$$

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:5px 0; padding-right:34px; line-height:1.85;">

‏▪ لو تسرعت وقرأت الأس قبل التبسيط لقلت الدرجة الأولى! ولكن لوجود الأس الكسري ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${\frac{1}{2}}$</span>‏ (جذر تربيعي) على المشتقة، <b>يجب تربيع الطرفين أولاً</b>:

</div>

<div dir="ltr" align="center" style="direction:ltr; text-align:center; margin:12px 0; overflow-x:auto;">

$$\left(\frac{\partial^2 z}{\partial x^2}\right)^2 = 1 + \frac{\partial z}{\partial y}$$

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:5px 0; padding-right:34px; line-height:1.85;">

‏▪ الآن نقرأ أعلى مشتقة وهي ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${\frac{\partial^2 z}{\partial x^2}}$</span>‏: <b>الرتبة = 2 (الثانية)</b>، ولأنها مرفوعة للأس 2 بعد التربيع فإن <b>الدرجة = 2 (الثانية)</b>!

</div>

</div>


<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; width:100%; max-width:100%; box-sizing:border-box; line-height:1.85; padding:6px 14px 16px 14px;">

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:12px 0 6px 0;">

### ‏ثالثاً: التصنيف الرباعي لمعادلات الرتبة الأولى (القسم 1.1.7 — ص 4–5 بالمذكرة / PDF ص 5–6)

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:6px 0; line-height:1.85;">

‏كيف تفرق بين الأنواع الأربعة لمعادلات الرتبة الأولى ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${f(x, y, z, p, q) = 0}$</span>‏ بمجرد النظر؟

</div>

<div dir="rtl" style="direction:rtl; text-align:right; overflow-x:auto; width:100%; margin:14px 0;">
<table dir="rtl" style="display:table; width:100%; border-collapse:collapse; direction:rtl; text-align:right;">
<thead>
<tr>
<th dir="rtl" align="right" style="direction:rtl; text-align:right; padding:9px 12px; border:1px solid rgba(128,128,128,0.4); background-color:rgba(128,128,128,0.16); font-weight:bold; line-height:1.75; vertical-align:middle;">

‏النوع في المذكرة (PDF ص 5–6)

</th>
<th dir="rtl" align="right" style="direction:rtl; text-align:right; padding:9px 12px; border:1px solid rgba(128,128,128,0.4); background-color:rgba(128,128,128,0.16); font-weight:bold; line-height:1.75; vertical-align:middle;">

‏الصورة العامة

</th>
<th dir="rtl" align="right" style="direction:rtl; text-align:right; padding:9px 12px; border:1px solid rgba(128,128,128,0.4); background-color:rgba(128,128,128,0.16); font-weight:bold; line-height:1.75; vertical-align:middle;">

‏الشرط المميز في خطوة واحدة

</th>
<th dir="rtl" align="right" style="direction:rtl; text-align:right; padding:9px 12px; border:1px solid rgba(128,128,128,0.4); background-color:rgba(128,128,128,0.16); font-weight:bold; line-height:1.75; vertical-align:middle;">

‏أمثلة المذكرة (PDF ص 5–6)

</th>
</tr>
</thead>
<tbody>
<tr>
<td dir="rtl" align="right" style="direction:rtl; text-align:right; padding:8px 12px; border:1px solid rgba(128,128,128,0.35); vertical-align:middle; line-height:1.8;">

‏<b>1. خطية تماماً (Linear)</b>

</td>
<td dir="rtl" align="right" style="direction:rtl; text-align:right; padding:8px 12px; border:1px solid rgba(128,128,128,0.35); vertical-align:middle; line-height:1.8;">

‏‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate; white-space:nowrap;">${P(x,y)p + Q(x,y)q = R(x,y)z + S(x,y)}$</span>‏

</td>
<td dir="rtl" align="right" style="direction:rtl; text-align:right; padding:8px 12px; border:1px solid rgba(128,128,128,0.35); vertical-align:middle; line-height:1.8;">

‏المشتقتان ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate; white-space:nowrap;">${p, q}$</span>‏ <b>وكذلك ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate; white-space:nowrap;">${z}$</span>‏</b> كلها من الدرجة الأولى، ولا يوجد ضرب بينها (‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate; white-space:nowrap;">${zp}$</span>‏ ممنوع!)

</td>
<td dir="rtl" align="right" style="direction:rtl; text-align:right; padding:8px 12px; border:1px solid rgba(128,128,128,0.35); vertical-align:middle; line-height:1.8;">

‏‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate; white-space:nowrap;">${yx^2 p + xy^2 q = xyz + x^2 y^3}$</span>‏

</td>
</tr>
<tr>
<td dir="rtl" align="right" style="direction:rtl; text-align:right; padding:8px 12px; border:1px solid rgba(128,128,128,0.35); vertical-align:middle; line-height:1.8;">

‏<b>2. نصف خطية (Semi-linear)</b>

</td>
<td dir="rtl" align="right" style="direction:rtl; text-align:right; padding:8px 12px; border:1px solid rgba(128,128,128,0.35); vertical-align:middle; line-height:1.8;">

‏‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate; white-space:nowrap;">${P(x,y)p + Q(x,y)q = R(x,y,z)}$</span>‏

</td>
<td dir="rtl" align="right" style="direction:rtl; text-align:right; padding:8px 12px; border:1px solid rgba(128,128,128,0.35); vertical-align:middle; line-height:1.8;">

‏المشتقتان ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate; white-space:nowrap;">${p, q}$</span>‏ من الدرجة الأولى، ومعاملاتهما ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate; white-space:nowrap;">${P(x,y), Q(x,y)}$</span>‏ <b>تعتمد على ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate; white-space:nowrap;">${x, y}$</span>‏ فقط</b> (خالية من ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate; white-space:nowrap;">${z}$</span>‏)، بينما الطرف الأيمن ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate; white-space:nowrap;">${R}$</span>‏ قد يكون غير خطي في ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate; white-space:nowrap;">${z}$</span>‏

</td>
<td dir="rtl" align="right" style="direction:rtl; text-align:right; padding:8px 12px; border:1px solid rgba(128,128,128,0.35); vertical-align:middle; line-height:1.8;">

‏‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate; white-space:nowrap;">${xyp + x^2 y q = x^2 y^2 z^2}$</span>‏

</td>
</tr>
<tr>
<td dir="rtl" align="right" style="direction:rtl; text-align:right; padding:8px 12px; border:1px solid rgba(128,128,128,0.35); vertical-align:middle; line-height:1.8;">

‏<b>3. شبه خطية (Quasi-linear)</b>

</td>
<td dir="rtl" align="right" style="direction:rtl; text-align:right; padding:8px 12px; border:1px solid rgba(128,128,128,0.35); vertical-align:middle; line-height:1.8;">

‏‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate; white-space:nowrap;">${P(x,y,z)p + Q(x,y,z)q = R(x,y,z)}$</span>‏

</td>
<td dir="rtl" align="right" style="direction:rtl; text-align:right; padding:8px 12px; border:1px solid rgba(128,128,128,0.35); vertical-align:middle; line-height:1.8;">

‏المشتقتان ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate; white-space:nowrap;">${p, q}$</span>‏ من الدرجة الأولى، ولكن معاملاتهما ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate; white-space:nowrap;">${P, Q}$</span>‏ <b>تحتوي على ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate; white-space:nowrap;">${z}$</span>‏</b> (هذه المعادلة وما سبقها تُحل جميعاً <b>بطريقة لاجرانج</b> في اللاب 2!)

</td>
<td dir="rtl" align="right" style="direction:rtl; text-align:right; padding:8px 12px; border:1px solid rgba(128,128,128,0.35); vertical-align:middle; line-height:1.8;">

‏‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate; white-space:nowrap;">${x^2 z p + y^2 z q = xy}$</span>‏

</td>
</tr>
<tr>
<td dir="rtl" align="right" style="direction:rtl; text-align:right; padding:8px 12px; border:1px solid rgba(128,128,128,0.35); vertical-align:middle; line-height:1.8;">

‏<b>4. غير خطية (Non-linear)</b>

</td>
<td dir="rtl" align="right" style="direction:rtl; text-align:right; padding:8px 12px; border:1px solid rgba(128,128,128,0.35); vertical-align:middle; line-height:1.8;">

‏‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate; white-space:nowrap;">${f(x,y,z,p,q) = 0}$</span>‏

</td>
<td dir="rtl" align="right" style="direction:rtl; text-align:right; padding:8px 12px; border:1px solid rgba(128,128,128,0.35); vertical-align:middle; line-height:1.8;">

‏تحتوي على تربيع ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate; white-space:nowrap;">${p^2, q^2}$</span>‏ أو حاصل ضرب ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate; white-space:nowrap;">${p \cdot q}$</span>‏ (وتُحل <b>بطريقة شاربت والأشكال الأربعة</b> في اللاب 3!)

</td>
<td dir="rtl" align="right" style="direction:rtl; text-align:right; padding:8px 12px; border:1px solid rgba(128,128,128,0.35); vertical-align:middle; line-height:1.8;">

‏‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate; white-space:nowrap;">${p^2 + q^2 = 1}$</span>‏ أو ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate; white-space:nowrap;">${pq = z}$</span>‏

</td>
</tr>
</tbody>
</table>
</div>

</div>


<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; width:100%; max-width:100%; box-sizing:border-box; line-height:1.85; padding:6px 14px 16px 14px;">

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:12px 0 6px 0;">

### ‏رابعاً: الحالات الثلاث لحذف الثوابت الاختيارية (القسم 1.2 — ص 6–9 بالمذكرة / PDF ص 7–10)

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:6px 0; line-height:1.85;">

‏عند حذف ثوابت اختيارية من علاقة ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${F(x, y, z, a, b) = 0}$</span>‏، نشتق جزئياً بالنسبة لـ ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${x}$</span>‏ وبالنسبة لـ ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${y}$</span>‏، وتنشأ إحدى <b>الحالات الثلاث التالية</b>:

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:7px 0; padding-right:8px; line-height:1.85;">

‏<b>1.</b> <b>الحالة (1) — عدد الثوابت ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${<}$</span>‏ عدد المتغيرات المستقلة (PDF ص 7–8):</b>

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:5px 0; padding-right:34px; line-height:1.85;">

‏▪ <b>النتيجة:</b> نحصل على <b>أكثر من معادلة تفاضلية جزئية</b> من الرتبة الأولى.

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:5px 0; padding-right:34px; line-height:1.85;">

‏▪ <b>مثال المذكرة (PDF ص 7):</b> ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${z = ax + y}$</span>‏ (ثابت واحد ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${a}$</span>‏ ومتغيران ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${x, y}$</span>‏):

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:5px 0; padding-right:34px; line-height:1.85;">

‏▪ نشتق بالنسبة لـ ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${x}$</span>‏: ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${\frac{\partial z}{\partial x} = a \implies}$</span>‏ نعوض عن ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${a}$</span>‏ في الأصلية فنحصل على المعادلة الأولى: ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${z = x\frac{\partial z}{\partial x} + y}$</span>‏.

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:5px 0; padding-right:34px; line-height:1.85;">

‏▪ نشتق بالنسبة لـ ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${y}$</span>‏: ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${\frac{\partial z}{\partial y} = 1}$</span>‏ (وهي خالية من ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${a}$</span>‏ أصلاً، فتمثل معادلة ثانية!).

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:7px 0; padding-right:8px; line-height:1.85;">

‏<b>2.</b> <b>الحالة (2) — عدد الثوابت ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${=}$</span>‏ عدد المتغيرات المستقلة (PDF ص 8):</b>

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:5px 0; padding-right:34px; line-height:1.85;">

‏▪ <b>النتيجة:</b> نحصل على <b>معادلة تفاضلية جزئية واحدة وحيدة</b> من الرتبة الأولى.

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:5px 0; padding-right:34px; line-height:1.85;">

‏▪ <b>مثال المذكرة (PDF ص 8):</b> احذف ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${a, b}$</span>‏ من ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${az + b = a^2 x + y}$</span>‏:

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:5px 0; padding-right:34px; line-height:1.85;">

‏▪ نشتق بالنسبة لـ ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${x}$</span>‏: ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${a\frac{\partial z}{\partial x} = a^2 \implies \frac{\partial z}{\partial x} = a}$</span>‏.

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:5px 0; padding-right:34px; line-height:1.85;">

‏▪ نشتق بالنسبة لـ ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${y}$</span>‏: ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${a\frac{\partial z}{\partial y} = 1}$</span>‏.

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:5px 0; padding-right:34px; line-height:1.85;">

‏▪ نعوض عن ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${a = \frac{\partial z}{\partial x}}$</span>‏ في المعادلة الثانية فنحصل فوراً على: ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${\left(\frac{\partial z}{\partial x}\right)\left(\frac{\partial z}{\partial y}\right) = 1}$</span>‏ (أي ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${pq = 1}$</span>‏).

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:7px 0; padding-right:8px; line-height:1.85;">

‏<b>3.</b> <b>الحالة (3) — عدد الثوابت ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${>}$</span>‏ عدد المتغيرات المستقلة (PDF ص 8–9):</b>

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:5px 0; padding-right:34px; line-height:1.85;">

‏▪ <b>النتيجة:</b> المشتقات الأولى وحدها لا تكفي لحذف جميع الثوابت، فنشتق مرة ثانية للحصول على معادلات من <b>الرتبة الثانية أو أعلى</b>.

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:5px 0; padding-right:34px; line-height:1.85;">

‏▪ <b>مثال المذكرة (PDF ص 9):</b> احذف الثوابت الثلاثة ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${a, b, c}$</span>‏ من ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${z = ax + by + cxy}$</span>‏:

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:5px 0; padding-right:34px; line-height:1.85;">

‏▪ المشتقات الأولى: ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${\frac{\partial z}{\partial x} = a + cy \;\cdots(2)}$</span>‏، و ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${\frac{\partial z}{\partial y} = b + cx \;\cdots(3)}$</span>‏.

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:5px 0; padding-right:34px; line-height:1.85;">

‏▪ المشتقات الثانية: ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${\frac{\partial^2 z}{\partial x^2} = 0}$</span>‏، و ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${\frac{\partial^2 z}{\partial y^2} = 0 \;\cdots(4)}$</span>‏، والمشتقة المختلطة: ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${\frac{\partial^2 z}{\partial x \partial y} = c \;\cdots(5)}$</span>‏.

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:5px 0; padding-right:34px; line-height:1.85;">

‏▪ بضرب ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${(2)}$</span>‏ في ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${x}$</span>‏ وضرب ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${(3)}$</span>‏ في ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${y}$</span>‏ وجمعهما:

</div>

<div dir="ltr" align="center" style="direction:ltr; text-align:center; margin:12px 0; overflow-x:auto;">

$$x\frac{\partial z}{\partial x} + y\frac{\partial z}{\partial y} = \underbrace{ax + by + cxy}_{=\,z} + cxy = z + xy\frac{\partial^2 z}{\partial x \partial y} \;\cdots(6)$$

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; border-right:4px solid #f59e0b; background-color:rgba(245,158,11,0.08); padding:10px 14px; margin:12px 0; border-radius:4px; line-height:1.85;">

‏⚠️ <b>تنبيه خطأ مطبعي في مذكرة الكلية (PDF صفحة 9 — المعادلتان 5 و 6):</b><br>
‏في ملف الـ PDF (صفحة 9، السطر 6 والسطر 11)، كُتبت المشتقة المختلطة بالخطأ المطبعي ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${\frac{\partial^2 y}{\partial x \partial y}}$</span>‏ (بحرف ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${y}$</span>‏ في البسط)، والصواب رياضياً هو ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${\frac{\partial^2 z}{\partial x \partial y} = c}$</span>‏ لأن ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${z}$</span>‏ هو المتغير التابع بينما ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${x, y}$</span>‏ متغيران مستقلان!

</div>

</div>


<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; width:100%; max-width:100%; box-sizing:border-box; line-height:1.85; padding:6px 14px 16px 14px;">

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:12px 0 6px 0;">

### ‏📖 مثال محلول بالتفصيل من المذكرة (Example 1 — صفحة 8 بالمذكرة / PDF ص 9–10):

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:6px 0; line-height:1.85;">

‏<b>المسألة:</b> كوّن المعادلة التفاضلية الجزئية بحذف الثابتين الاختياريين ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${a}$</span>‏ و ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${b}$</span>‏ من عائلة المستويات:

</div>

<div dir="ltr" align="center" style="direction:ltr; text-align:center; margin:12px 0; overflow-x:auto;">

$$z = a x + b y + a^2 + b^2 \qquad \cdots (1)$$

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:6px 0; line-height:1.85;">

‏<b>الحل خطوة بخطوة (مع سبب كل خطوة):</b>

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:5px 0; padding-right:14px; line-height:1.85;">

‏• <b>الخطوة 1 (الاشتقاق بالنسبة لـ ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${x}$</span>‏):</b> نعتبر ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${y}$</span>‏ ثابتاً ونشتق المعادلة ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${(1)}$</span>‏ بالنسبة لـ ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${x}$</span>‏:

</div>

<div dir="ltr" align="center" style="direction:ltr; text-align:center; margin:12px 0; overflow-x:auto;">

$$p = \frac{\partial z}{\partial x} = a \qquad \cdots (2)$$

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:6px 0; line-height:1.85;">

‏<i>(لماذا؟ لأن ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${by + a^2 + b^2}$</span>‏ كلها ثوابت بالنسبة لـ ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${x}$</span>‏ فمشتقتها صفر!)</i>

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:5px 0; padding-right:14px; line-height:1.85;">

‏• <b>الخطوة 2 (الاشتقاق بالنسبة لـ ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${y}$</span>‏):</b> نعتبر ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${x}$</span>‏ ثابتاً ونشتق المعادلة ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${(1)}$</span>‏ بالنسبة لـ ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${y}$</span>‏:

</div>

<div dir="ltr" align="center" style="direction:ltr; text-align:center; margin:12px 0; overflow-x:auto;">

$$q = \frac{\partial z}{\partial y} = b \qquad \cdots (3)$$

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:5px 0; padding-right:14px; line-height:1.85;">

‏• <b>الخطوة 3 (التعويض والحذف):</b> نعوض عن ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${a = p}$</span>‏ وعن ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${b = q}$</span>‏ في المعادلة الأصلية ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${(1)}$</span>‏:

</div>

<div dir="ltr" align="center" style="direction:ltr; text-align:center; margin:12px 0; overflow-x:auto;">

$$z = x p + y q + p^2 + q^2$$

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:5px 0; padding-right:14px; line-height:1.85;">

‏• <b>التحقق السريع:</b> هل بقيت أي رموز ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${a}$</span>‏ أو ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${b}$</span>‏؟ لا! إذن حصلنا على معادلة تفاضلية جزئية غير خطية من الرتبة الأولى والدرجة الثانية (وهي على صيغة كليروت التي سنحلها في اللاب 3!).

</div>

</div>


<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; width:100%; max-width:100%; box-sizing:border-box; line-height:1.85; padding:6px 14px 16px 14px;">

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:12px 0 6px 0;">

## ‏3. ✍️ تحدي الورقة والقلم 1 (Example 2 — صفحة 9 بالمذكرة / PDF ص 10)

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:6px 0; line-height:1.85;">

‏<b>المسألة:</b> احذف الثابتين الاختياريين ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${a}$</span>‏ و ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${b}$</span>‏ من معادلة عائلة الأسطح المكافئة الدائرية (حفر المريخ):

</div>

<div dir="ltr" align="center" style="direction:ltr; text-align:center; margin:12px 0; overflow-x:auto;">

$$z = (x - a)^2 + (y - b)^2 \qquad \cdots (1)$$

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:6px 0; line-height:1.85;">

‏<b>جرّب الحل بنفسك على الورقة أولاً!</b> إذا احتجت مساعدة، افتح التلميحات التدريجية التالية:

</div>

<details dir="rtl" style="direction:rtl; text-align:right; margin:8px 0; padding:6px 10px;">

<summary dir="rtl" style="direction:rtl; text-align:right; cursor:pointer; font-weight:bold;">

‏💡 <b>تلميح 1: كيف نبدأ؟</b> (اضغط للفتح)

</summary>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:6px 0; line-height:1.85;">

‏اشتق المعادلة ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${(1)}$</span>‏ جزئياً بالنسبة لـ ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${x}$</span>‏ لتحصل على ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${p}$</span>‏، ثم بالنسبة لـ ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${y}$</span>‏ لتحصل على ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${q}$</span>‏. تذكر قاعدة السلسلة: مشتقة ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${(x-a)^2}$</span>‏ بالنسبة لـ ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${x}$</span>‏ هي ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${2(x-a)}$</span>‏.

</div>

</details>

<details dir="rtl" style="direction:rtl; text-align:right; margin:8px 0; padding:6px 10px;">

<summary dir="rtl" style="direction:rtl; text-align:right; cursor:pointer; font-weight:bold;">

‏💡 <b>تلميح 2: كيف نتخلص من ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${(x-a)}$</span>‏ و ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${(y-b)}$</span>‏؟</b> (اضغط للفتح)

</summary>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:6px 0; line-height:1.85;">

‏أصبح لديك:

</div>

<div dir="ltr" align="center" style="direction:ltr; text-align:center; margin:12px 0; overflow-x:auto;">

$$p = 2(x - a) \quad \text{و} \quad q = 2(y - b)$$

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:6px 0; line-height:1.85;">

‏لاحظ أن المعادلة الأصلية ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${(1)}$</span>‏ تحتوي على ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${(x-a)^2 + (y-b)^2}$</span>‏. ماذا يحدث لو ربّعت ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${p}$</span>‏ وربّعت ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${q}$</span>‏ وجمعتهما؟

</div>

</details>

<details dir="rtl" style="direction:rtl; text-align:right; margin:8px 0; padding:6px 10px;">

<summary dir="rtl" style="direction:rtl; text-align:right; cursor:pointer; font-weight:bold;">

‏💡 <b>تلميح 3: الخطوة الأخيرة</b> (اضغط للفتح)

</summary>

<div dir="ltr" align="center" style="direction:ltr; text-align:center; margin:12px 0; overflow-x:auto;">

$$p^2 + q^2 = 4(x - a)^2 + 4(y - b)^2 = 4\underbrace{\left[(x - a)^2 + (y - b)^2\right]}_{=\,z}$$

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:6px 0; line-height:1.85;">

‏استبدل القوس الكبير بـ ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${z}$</span>‏ من المعادلة ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${(1)}$</span>‏.

</div>

</details>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:6px 0; line-height:1.85;">

‏اكتب المعادلة التفاضلية الجزئية الناتجة كنص في الخلية التالية (بصيغة مثل ‏<code dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate; padding:1px 5px; border-radius:3px;">"p^2 + q^2 = 4&#42;z"</code>‏):

</div>

</div>


In [ ]:
# خلية تقييم: تحدي الورقة 1
### START CODE HERE ###
student_pde_1 = "p^2 + q^2 = 4*z"
### END CODE HERE ###

# يفحص المصحح الآلي معادلتك بالتعويض العددي في عشرات النقاط العشوائية:
test_lab1.check_paper_challenge_1(student_pde_1)


<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; width:100%; max-width:100%; box-sizing:border-box; line-height:1.85; padding:6px 14px 16px 14px;">

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:12px 0 6px 0;">

## ‏4. 🚀 تحدي الربط بالمشروع: فحص حوض الهبوط الإهليلجي

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:6px 0; line-height:1.85;">

‏في تضاريس المريخ، قد تكون الحفرة ممتدة (إهليلجية) بمعامل انحناء ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${a}$</span>‏ شرقاً و ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${b}$</span>‏ شمالاً:

</div>

<div dir="ltr" align="center" style="direction:ltr; text-align:center; margin:12px 0; overflow-x:auto;">

$$z = \frac{1}{2} a x^2 + \frac{1}{2} b y^2 \qquad \cdots (1)$$

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:6px 0; line-height:1.85;">

‏<b>المطلوب على ورقتك:</b>

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:6px 0; line-height:1.85;">

‏احذف الثابتين الاختياريين ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${a}$</span>‏ و ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${b}$</span>‏ لتكوين المعادلة التفاضلية الجزئية التي تربط الارتفاع ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${z}$</span>‏ بالميول ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${(p, q)}$</span>‏ عند أي نقطة ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${(x, y)}$</span>‏.

</div>

<details dir="rtl" style="direction:rtl; text-align:right; margin:8px 0; padding:6px 10px;">

<summary dir="rtl" style="direction:rtl; text-align:right; cursor:pointer; font-weight:bold;">

‏💡 <b>تلميح 1:</b> احسب ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${p}$</span>‏ و ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${q}$</span>‏

</summary>

<div dir="ltr" align="center" style="direction:ltr; text-align:center; margin:12px 0; overflow-x:auto;">

$$p = \frac{\partial z}{\partial x} = a x \implies a = \frac{p}{x}$$

</div>

<div dir="ltr" align="center" style="direction:ltr; text-align:center; margin:12px 0; overflow-x:auto;">

$$q = \frac{\partial z}{\partial y} = b y \implies b = \frac{q}{y}$$

</div>

</details>

<details dir="rtl" style="direction:rtl; text-align:right; margin:8px 0; padding:6px 10px;">

<summary dir="rtl" style="direction:rtl; text-align:right; cursor:pointer; font-weight:bold;">

‏💡 <b>تلميح 2:</b> عوّض في المعادلة الأصلية

</summary>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:6px 0; line-height:1.85;">

‏عوّض عن ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${a = \frac{p}{x}}$</span>‏ وعن ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${b = \frac{q}{y}}$</span>‏ في المعادلة ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${(1)}$</span>‏ وبسّط الكسور (يمكنك ضرب الطرفين في ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${2}$</span>‏ للتخلص من الكسور).

</div>

</details>

</div>


In [ ]:
# خلية تقييم: تحدي حوض المريخ
### START CODE HERE ###
rover_basin_pde = "x*p + y*q = 2*z"
### END CODE HERE ###

test_lab1.check_paper_challenge_2(rover_basin_pde)


<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; width:100%; max-width:100%; box-sizing:border-box; line-height:1.85; padding:6px 14px 16px 14px;">

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:12px 0 6px 0;">

## ‏5. 💻 من الورقة إلى الكود: التحقق العددي على شبكة التضاريس

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:6px 0; line-height:1.85;">

‏<b>كيف نتحقق برمجياً من أن حلنا الورقي صحيح؟</b>

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:6px 0; line-height:1.85;">

‏إذا كانت المعادلة ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${p^2 + q^2 = 4z}$</span>‏ صحيحة، فإن الفرق بين الطرف الأيسر والطرف الأيمن (ويسمى <b>المتبقي Residual</b>):

</div>

<div dir="ltr" align="center" style="direction:ltr; text-align:center; margin:12px 0; overflow-x:auto;">

$$\text{residual} = (p^2 + q^2) - 4z$$

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:6px 0; line-height:1.85;">

‏يجب أن يساوي <b>صفراً</b> عند كل نقطة من نقاط الشبكة، أياً كان مركز الحفرة ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${(a, b)}$</span>‏!

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:6px 0; line-height:1.85;">

‏أكمل الدالة ‏<code dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate; padding:1px 5px; border-radius:3px;">verify&#95;crater&#95;pde</code>‏ أدناه لحساب مصفوفة المتبقي:

</div>

</div>


In [ ]:
# دالة تقييم برمجية: verify_crater_pde
def verify_crater_pde(Z: np.ndarray, P: np.ndarray, Q: np.ndarray) -> np.ndarray:
    """تحسب المتبقي (p^2 + q^2) - 4*z للتحقق من معادلة الحفرة على كامل الشبكة."""
    ### START CODE HERE ###
    residual = (P**2 + Q**2) - 4.0 * Z
    ### END CODE HERE ###
    return residual

test_lab1.test_eliminate_constants_fn(verify_crater_pde)


<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; width:100%; max-width:100%; box-sizing:border-box; line-height:1.85; padding:6px 14px 16px 14px;">

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:12px 0 6px 0;">

## ‏6. 🛰️ الرؤية البصرية والربط الفعلي بكود المشروع (‏<code dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate; padding:1px 5px; border-radius:3px;">terrain.slope&#95;map</code>‏)

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:6px 0; line-height:1.85;">

‏شغّل الخلية التالية لترى شيئين معاً:

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:7px 0; padding-right:8px; line-height:1.85;">

‏<b>1.</b> <b>على اليسار والوسط:</b> سطح الحفرة ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${z=(x-a)^2+(y-b)^2}$</span>‏ ومتجهات الميل ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${(p, q)}$</span>‏ التي حسبتها على الورق.

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:7px 0; padding-right:8px; line-height:1.85;">

‏<b>2.</b> <b>على اليمين (من محرك المشروع الفعلي ‏<code dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate; padding:1px 5px; border-radius:3px;">rover&#95;path&#95;finder/terrain.py</code>‏):</b> كيف تستخدم دالة ‏<code dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate; padding:1px 5px; border-radius:3px;">slope&#95;map</code>‏ نفس المشتقتين ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${(p, q)}$</span>‏ لحساب زاوية الميل بالدرجات ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${\theta = \arctan\sqrt{p^2+q^2}}$</span>‏ وتحديد المناطق الخطرة (الحمراء ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${> 20^\circ}$</span>‏) على تضاريس المريخ!

</div>

</div>


In [ ]:
from rover_path_finder.terrain import make_terrain, slope_map

x = np.linspace(-3, 3, 40)
y = np.linspace(-3, 3, 40)
X, Y = np.meshgrid(x, y)
Z = (X - 0.5)**2 + (Y + 0.5)**2  # حفرة مركزها (a=0.5, b=-0.5)

dx = x[1] - x[0]
dy = y[1] - y[0]
# انتبه: المحور 0 هو y (الصفوف) والمحور 1 هو x (الأعمدة)
Q, P = np.gradient(Z, dy, dx)

# استدعاء دالة المشروع الحقيقية لحساب خريطة الميول على تضاريس المريخ
mars_demo = make_terrain(N=100, seed=3)
mars_slopes_deg = slope_map(mars_demo["h"], cell=1.0)

fig = plt.figure(figsize=(16, 4.8))

ax1 = fig.add_subplot(1, 3, 1, projection='3d')
ax1.plot_surface(X, Y, Z, cmap='copper', alpha=0.85)
ax1.set_title("1. Crater Surface z = (x-a)^2 + (y-b)^2", fontsize=11)
ax1.set_xlabel("x")
ax1.set_ylabel("y")

ax2 = fig.add_subplot(1, 3, 2)
ax2.contour(X, Y, Z, levels=15, cmap='copper')
ax2.quiver(X[::3, ::3], Y[::3, ::3], P[::3, ::3], Q[::3, ::3], color='crimson')
ax2.set_title(r"2. Slope Vectors (p, q) = ($\partial z/\partial x$, $\partial z/\partial y$)", fontsize=11)
ax2.set_xlabel("x")
ax2.set_ylabel("y")

ax3 = fig.add_subplot(1, 3, 3)
im = ax3.imshow(mars_slopes_deg, cmap='inferno', origin='lower')
ax3.contour(mars_demo["raw"], levels=[0.5], colors='cyan', linewidths=1.2)
plt.colorbar(im, ax=ax3, label="Slope (degrees)")
ax3.set_title("3. Project Engine: terrain.slope_map() & Hazards (cyan)", fontsize=11)
ax3.set_xlabel("Grid X")
ax3.set_ylabel("Grid Y")

plt.tight_layout()
plt.show()

#📌 ملخص اللاب 1 في 3 نقاط:
print("📌 ملخص اللاب 1:")
print("  1) p = dz/dx و q = dz/dy هما مركبتا ميل التضاريس في اتجاهي x و y.")
print("  2) حذف الثوابت الاختيارية (a, b) يعطينا قانوناً عاماً للميل (p^2 + q^2 = 4z) لا يتوقف على موقع الحفرة.")
print("  3) دالة slope_map() في المشروع تطبق هذا القانون لتحويل خريطة الارتفاعات إلى قناع عوائق (Hazard Mask).")
